In [ ]:
# CELL 0 - Environment Check
# ─────────────────────────────────────────────────────────────────────────────
# PrecisionSkin - ACNE BOXING SPRINT (same layout as the acne sprint).
# Goal: box a sample of acne photos by hand (one clear rule) so the detector learns from consistent boxes, and keep a part of them as a hand-boxed TEST set.
#
#   CELL 2  picks the photos (always run it): N_PHOTOS in total, N_HAND_TEST of them become the TEST set (never trained on)
#   CELL 3 packages -> CELL 4 progress -> CELL 5 collect (two folders: training boxes and test boxes)
#
# Your original photos are never changed (copies only). Nothing is uploaded by this notebook.
# Kernel: any WSL kernel with pandas / Pillow / OpenCV (the .venv works).
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, json, time, shutil, random, hashlib, zipfile, datetime
from pathlib import Path
import numpy as np, pandas as pd, cv2, yaml
from PIL import Image
print("Python:", sys.version.split()[0], "| kernel:", sys.executable)
print("✓ imports OK")

In [ ]:
# CELL 1 - Configuration
# ─────────────────────────────────────────────────────────────────────────────
#   ANNOTATORS         : who boxes with the LOCAL route (no spaces). One name = you alone.
#   N_PHOTOS / N_HAND_TEST : total photos to box / how many of them are kept as the TEST set.
#   N_CALIBRATION      : photos that EVERYONE boxes, to measure agreement (about 20; use 0 if you are alone).
#   SPRINT_ROOT        : where the per-person folders and zips are made (Windows side, so the tool can open them).
#   COLLECTED_DIR      : the finished YOLO folder the detector notebook reads (LOCAL route).
# ─────────────────────────────────────────────────────────────────────────────
ANNOTATORS = ["Elaiza"]
N_CALIBRATION = 0
N_PHOTOS = 600          # how many acne photos in total (acne photos have MANY spots: about 10 per photo; 600 photos is a big job: you can stop at any time, saved photos are kept)
N_HAND_TEST = 100       # of those, this many are the hand-boxed TEST set: they are never used for training
SEED = 42

REPO = Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp")
CANDIDATES_CSV = REPO / "acne_facial_candidates.csv"
# where your classifier photos are (the folder that holds Acne / Normal / ...). The first existing one is used, so moving the folder is fine.
_CLASSES_CANDIDATES = [Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Classes"), Path("/mnt/c/Users/Bautista/Downloads/Classes")]
CLASSES_ROOT = next((p for p in _CLASSES_CANDIDATES if (p / "Acne").exists()), _CLASSES_CANDIDATES[0])
print("classifier photos folder:", CLASSES_ROOT, "| found:", (CLASSES_ROOT / "Acne").exists())
SPRINT_ROOT = Path("/mnt/c/Users/Bautista/Documents/PrecisionSkin_DetectorOwn/acne_sprint")
COLLECTED_DIR = Path("/mnt/c/Users/Bautista/Downloads/own_acne.yolov8")
COLLECTED_TEST_DIR = Path("/mnt/c/Users/Bautista/Downloads/own_acne_TEST.yolov8")      # hand-boxed TEST photos (never trained on)
TEST_SELECTION_CSV = Path.home() / "PrecisionSkinV1" / "detector_own_images" / "own_images_v3" / "selection_lesion_photos.csv"    # optional: photos already used in the hand-boxed TEST set
CONDITION = "acne"
NEAR_DUP_CORR = 0.97

SPRINT_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def to_win(p):
    s = str(p)
    return ("C:\\" + s[len("/mnt/c/"):].replace("/", "\\")) if s.startswith("/mnt/c/") else s

def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()

print("annotators:", ANNOTATORS, "| calibration photos:", N_CALIBRATION)
print("sprint folder:", SPRINT_ROOT)
print("✓ configuration set")

In [ ]:
# CELL 2 - Pick the Photos to Box
# ─────────────────────────────────────────────────────────────────────────────
# From your candidate list: facial acne photos that are meant for TRAINING (not the ones reserved for the hand-boxed test set),
# that exist on disk, and that are not duplicates: same source photo, identical file, or near-identical picture (also rotated / flipped copies).
# The photos that already have PUBLIC boxes are INCLUDED: your boxes will replace the public ones (the detector notebook does the replacing).
# ─────────────────────────────────────────────────────────────────────────────
cand = pd.read_csv(CANDIDATES_CSV)
cand["src_path"] = [str(CLASSES_ROOT / Path(str(p)).parent.name / Path(str(p)).name) for p in cand.local_path_windows]      # same folder name + file name, under CLASSES_ROOT
n0 = len(cand)
pool = cand[cand.role_for_detector.str.startswith("pseudo-label")]
print("candidate list: %d photos | meant for training/validation: %d | reserved for the test set (not used here): %d" % (n0, len(pool), n0 - len(pool)))
pool = pool[pool.src_path.map(lambda p: Path(p).exists())].drop_duplicates("source_photo_key")
test_md5 = set(pd.read_csv(TEST_SELECTION_CSV).md5) if TEST_SELECTION_CSV.exists() else set()

keep, seen_md5 = [], set(test_md5)
for r in pool.itertuples():
    h = md5_of(r.src_path)
    if h not in seen_md5:
        seen_md5.add(h); keep.append((r, h))
print("after removing same-source and identical files: %d" % len(keep))

def thumb(path):
    im = cv2.imdecode(np.fromfile(path, np.uint8), cv2.IMREAD_GRAYSCALE)
    return None if im is None else cv2.resize(im, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
def zscore(x):
    x = x.reshape(len(x), -1); x = x - x.mean(1, keepdims=True)
    return (x / (x.std(1, keepdims=True) + 1e-6) / np.sqrt(x.shape[1])).astype(np.float32)

th = [thumb(r.src_path) for r, _ in keep]
ok = [i for i, t in enumerate(th) if t is not None]
G = np.stack([th[i] for i in ok])
base = zscore(G); best = np.full((len(ok), len(ok)), -1.0, np.float32)
for k in range(4):
    for flip in (False, True):
        v = np.rot90(G, k, axes=(1, 2)); v = v[:, :, ::-1] if flip else v
        best = np.maximum(best, base @ zscore(np.ascontiguousarray(v)).T)
dropped, chosen = set(), []
for a in range(len(ok)):
    if a in dropped:
        continue
    chosen.append(ok[a])
    for b in np.where(best[a] >= NEAR_DUP_CORR)[0]:
        if b > a:
            dropped.add(int(b))
print("near-identical copies dropped: %d" % len(dropped))

rows = []
for n, i in enumerate(chosen):
    r, h = keep[i]
    rows.append(dict(name="%s_%04d.jpg" % (CONDITION, n), src_path=r.src_path, md5=h, image_type=r.image_type, has_public_boxes=str(r.boxed_already), source_photo_key=r.source_photo_key))
sel = pd.DataFrame(rows)
# random sample: N_PHOTOS in total, the first N_HAND_TEST become the hand-boxed TEST set
sel = sel.sample(frac=1, random_state=SEED).reset_index(drop=True)
if N_PHOTOS and len(sel) > N_PHOTOS:
    sel = sel.iloc[:N_PHOTOS].copy()
sel["role"] = ["hand_test" if k < N_HAND_TEST else "train" for k in range(len(sel))]
sel.to_csv(SPRINT_ROOT / "sprint_photos.csv", index=False)
print("\nPHOTOS TO BOX: %d  (training %d, hand-boxed TEST %d)" % (len(sel), (sel.role == "train").sum(), (sel.role == "hand_test").sum()))
print(sel.image_type.value_counts().to_string())
print("with public boxes already (your boxes will replace them): %d | brand new: %d" % ((sel.has_public_boxes == "True").sum(), (sel.has_public_boxes != "True").sum()))

In [ ]:
# CELL 3 - LOCAL ROUTE: Share Out the Photos and Make the Packages
# ─────────────────────────────────────────────────────────────────────────────
# Each person gets a folder <Name>__acne with their photos. The calibration photos go to everyone.
# Packages: send_to_<Name>.zip = their photos + the box tool + START_HERE.txt (the exact commands and the labelling rule).
# THE RULE (talk it through together before starting):
#   one TIGHT box per spot (papule, pustule, comedo, nodule): the whole raised/red spot, not the skin around it.
#   Touching spots get separate boxes when you can tell them apart. Do not box scars, freckles, moles or dark marks. No visible acne: press X.
# ─────────────────────────────────────────────────────────────────────────────
rng = random.Random(SEED)
names = sel.name.tolist(); rng.shuffle(names)
calib, rest = names[:N_CALIBRATION], names[N_CALIBRATION:]
assign = [dict(name=n, annotator=ANNOTATORS[i % len(ANNOTATORS)], calibration=False) for i, n in enumerate(rest)] + [dict(name=n, annotator="ALL", calibration=True) for n in calib]
assign = pd.DataFrame(assign); assign.to_csv(SPRINT_ROOT / "sprint_assignments.csv", index=False)
src_of = dict(zip(sel.name, sel.src_path))

meta = SPRINT_ROOT / "_drafts" / "meta.json"
(SPRINT_ROOT / "_drafts").mkdir(exist_ok=True)
if not meta.exists():
    json.dump({"drafted_at": time.time(), "note": "sprint folders created"}, open(meta, "w"))

def folder_of(a):
    return SPRINT_ROOT / ("%s__%s" % (a, CONDITION))

tool_src = REPO / "lesion_box_tool.py"
for a in ANNOTATORS:
    d = folder_of(a); (d / "images").mkdir(parents=True, exist_ok=True); (d / "labels").mkdir(parents=True, exist_ok=True); (d / "classes.txt").write_text(CONDITION + "\n")
    for n in assign[(assign.annotator == a) | (assign.annotator == "ALL")].name:
        if not (d / "images" / n).exists():
            shutil.copy2(src_of[n], d / "images" / n)
    txt = ["HOW TO BOX YOUR ACNE PHOTOS", "",
           "1. You only need Python 3.8 or newer (python.org, tick 'Add python to PATH'). Nothing else to install.",
           "2. Unzip this package, open PowerShell in the unzipped folder and run:", "",
           '   py -3 lesion_box_tool.py "%s__%s" --label %s --annotator %s --port 8800' % (a, CONDITION, CONDITION, a), "",
           "3. A browser tab opens. Draw one TIGHT box around EACH acne spot (the whole raised/red spot, not the skin around it).",
           "   S = save + next photo | X = no acne visible | G = show the rule | V = review sheet | Ctrl+Z = undo",
           "   Mouse wheel = zoom, Space+drag = move the picture, E = contrast view, H = hide boxes.",
           "4. Use the SAME rule on every photo. Touching spots get separate boxes when you can tell them apart. Do not box scars, freckles, moles or dark marks.",
           "5. You can stop and continue later: saved photos stay saved. Press Ctrl+C in PowerShell to stop the tool.",
           "6. When finished (or once a day), zip the folder '%s__%s' and send it back. Do not rename any file." % (a, CONDITION)]
    zp = SPRINT_ROOT / ("send_to_%s.zip" % a)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("START_HERE.txt", "\n".join(txt)); z.write(tool_src, "lesion_box_tool.py"); z.write(meta, "_drafts/meta.json")
        for f in d.rglob("*"):
            if f.is_file():
                z.write(f, str(f.relative_to(SPRINT_ROOT)))
    n_mine = int(((assign.annotator == a) | (assign.annotator == "ALL")).sum())
    print("package ready: %-26s %4d photos, %.0f MB" % (zp.name, n_mine, zp.stat().st_size / 1e6))
print("\nYour own command (Windows PowerShell inside the SkinDevApp folder):")
print('  py -3.13 lesion_box_tool.py "%s" --label %s --annotator %s --port 8800' % (to_win(folder_of(ANNOTATORS[0])), CONDITION, ANNOTATORS[0]))
print("\nWhen the others send their folders back, put them into", SPRINT_ROOT, "(replace the matching folder) and run CELL 4.")

In [ ]:
# CELL 4 - LOCAL ROUTE: Progress per Person
# ─────────────────────────────────────────────────────────────────────────────
# How many photos each person has saved, how many were marked "no acne" (X), boxes per photo and seconds per photo (from the tool's log).
# Re-run any time. (Also available as:  py -3.13 box_progress.py "<sprint folder>")
# ─────────────────────────────────────────────────────────────────────────────
import csv, statistics as st
t_ref = json.load(open(SPRINT_ROOT / "_drafts" / "meta.json"))["drafted_at"]
assign = pd.read_csv(SPRINT_ROOT / "sprint_assignments.csv")
src_of = dict(zip(sel.name, sel.src_path))

def saved_label(a, name):
    p = folder_of(a) / "labels" / (Path(name).stem + ".txt")
    return p if (p.exists() and p.stat().st_mtime > t_ref + 1) else None

rows = []
for a in ANNOTATORS:
    mine = assign[(assign.annotator == a) | (assign.annotator == "ALL")].name.tolist()
    saved = [n for n in mine if saved_label(a, n)]
    nb = [sum(1 for l in saved_label(a, n).read_text().splitlines() if len(l.split()) == 5) for n in saved]
    secs = []
    lg = folder_of(a) / "labeling_log.csv"
    if lg.exists():
        last = {}
        for r in csv.DictReader(open(lg, encoding="utf-8")):
            try:
                if r.get("action") in ("save", "none"): last[r["image"]] = float(r["seconds"])
            except Exception: pass
        secs = list(last.values())
    rows.append(dict(person=a, saved="%d / %d" % (len(saved), len(mine)), no_patch_X=sum(1 for n in nb if n == 0), boxes_per_photo=round(float(np.mean([n for n in nb if n > 0])), 1) if any(nb) else None,
                     sec_per_photo=round(float(np.mean(secs))) if secs else None))
prog = pd.DataFrame(rows); print(prog.to_string(index=False))
tot = sum(int(r["saved"].split(" / ")[0]) for r in rows); allp = sum(int(r["saved"].split(" / ")[1]) for r in rows)
print("\nTOTAL saved %d of %d (calibration photos count once per person)" % (tot, allp))

In [ ]:
# CELL 5 - LOCAL ROUTE: Collect the Boxes into ONE YOLO Folder
# ─────────────────────────────────────────────────────────────────────────────
# Every photo uses the boxes of its assigned person (calibration photos: the first person who saved them). Photos marked "no acne" are left out. Photos with role hand_test go to COLLECTED_TEST_DIR (the TEST set), the others to COLLECTED_DIR.
# Writes COLLECTED_DIR/images, labels (class 0 = acne) and data.yaml. This folder goes into OWN_BOX_SOURCES in the detector notebook
# with "replace_public_twins": True. Also prints a quality summary and the AGREEMENT between people on the shared photos.
# ─────────────────────────────────────────────────────────────────────────────
def read_lab(p):
    return [tuple(float(v) for v in l.split()[1:]) for l in p.read_text().splitlines() if len(l.split()) == 5] if p else []

role_of = dict(zip(sel.name, sel.role))
for _d in (COLLECTED_DIR, COLLECTED_TEST_DIR):
    if _d.exists():
        shutil.rmtree(_d)
    (_d / "images").mkdir(parents=True); (_d / "labels").mkdir(parents=True)
n_used = {"train": 0, "hand_test": 0}; n_boxes = {"train": 0, "hand_test": 0}
used = saved_n = 0; qrows = []
for r in assign.itertuples():
    owners = ANNOTATORS if r.annotator == "ALL" else [r.annotator]
    got = [saved_label(a, r.name) for a in owners]; got = [g for g in got if g]
    if not got:
        continue
    saved_n += 1
    bx = read_lab(got[0])
    if not bx:
        continue
    used += 1
    dest = COLLECTED_TEST_DIR if role_of[r.name] == "hand_test" else COLLECTED_DIR
    n_used[role_of[r.name]] += 1; n_boxes[role_of[r.name]] += len(bx)
    shutil.copy2(src_of[r.name], dest / "images" / r.name)
    (dest / "labels" / (Path(r.name).stem + ".txt")).write_text("".join("0 %.6f %.6f %.6f %.6f\n" % b for b in bx))
    w_, h_ = Image.open(src_of[r.name]).size
    qrows += [dict(image=r.name, w_px=b[2] * w_, h_px=b[3] * h_, area_frac=b[2] * b[3]) for b in bx]
for _d in (COLLECTED_DIR, COLLECTED_TEST_DIR):
    yaml.safe_dump({"path": str(_d), "train": "images", "val": "images", "test": "images", "names": {0: CONDITION}}, open(_d / "data.yaml", "w"))
print("TRAINING boxes: %d photos / %d boxes  ->  %s" % (n_used["train"], n_boxes["train"], COLLECTED_DIR))
print("TEST boxes    : %d photos / %d boxes  ->  %s   (never train on these)" % (n_used["hand_test"], n_boxes["hand_test"], COLLECTED_TEST_DIR))
print("photos saved: %d of %d | used (at least one box): %d | boxes: %d" % (saved_n, len(assign), used, len(qrows)))
if qrows:
    q = pd.DataFrame(qrows)
    print("\nbox size (pixels): median %.0f x %.0f | middle 50%%: %.0f-%.0f wide" % (q.w_px.median(), q.h_px.median(), q.w_px.quantile(.25), q.w_px.quantile(.75)))
    print("boxes per photo: mean %.1f, median %.0f, max %d" % (q.groupby("image").size().mean(), q.groupby("image").size().median(), q.groupby("image").size().max()))
    med = q.area_frac.median(); odd = q[(q.area_frac > 6 * med) | (q.area_frac < med / 6)].image.unique()
    print("photos with an unusually large or small box (re-check with the review sheet, V): %d" % len(odd), list(odd[:10]))

def iou_xywh(a, b):
    ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw_, ih_ = max(0, min(ax1, bx1) - max(ax0, bx0)), max(0, min(ay1, by1) - max(ay0, by0)); i = iw_ * ih_
    u = a[2] * a[3] + b[2] * b[3] - i
    return i / u if u > 0 else 0.0
def pair_f1(A, B):
    used_, tp = set(), 0
    for x in A:
        best, bj = 0.0, None
        for j, y in enumerate(B):
            if j in used_: continue
            v = iou_xywh(x, y)
            if v > best: best, bj = v, j
        if bj is not None and best >= 0.5:
            used_.add(bj); tp += 1
    return 2 * tp / max(1, len(A) + len(B))
ag = []
for i, a1 in enumerate(ANNOTATORS):
    for a2 in ANNOTATORS[i + 1:]:
        f = [pair_f1(read_lab(saved_label(a1, n)), read_lab(saved_label(a2, n))) for n in assign[assign.annotator == "ALL"].name if saved_label(a1, n) and saved_label(a2, n)]
        if f: ag.append(dict(pair="%s vs %s" % (a1, a2), photos=len(f), agreement_F1=round(float(np.mean(f)), 3)))
print("\nAGREEMENT on the shared photos (F1 at IoU 0.5; about 0.6 or higher is reasonable):")
print(pd.DataFrame(ag).to_string(index=False) if ag else "not available yet (needs the shared photos boxed by at least two people)")
print("\nUse this in the detector notebook (CELL 2):")
print('  OWN_BOX_SOURCES: {"name": "acne_hand", "path": Path(r"%s"), "splits": [], "replace_public_twins": True},' % COLLECTED_DIR)
print('  hand-boxed TEST set: %s' % COLLECTED_TEST_DIR)